# Notebook 2: Data Cleaning & Feature Engineering
**Purpose:** Fix all data quality issues, handle missing/duplicate records, and engineer new columns.
**Inputs:** `data/raw/Online_Retail.csv`
**Outputs:** `data/processed/cleaned_sales.csv`, `data/processed/cancellations.csv`


In [1]:
# Import libraries and load data
import pandas as pd
import numpy as np

# Load the raw dataset
df = pd.read_csv('../data/raw/Online_Retail.csv', encoding='ISO-8859-1')
print(f"Raw rows: {len(df)}")


Raw rows: 541909


In [2]:
# Step 1 — Remove duplicates
# We drop exact duplicate rows to prevent double-counting revenue
raw_count = len(df)
df = df.drop_duplicates()
print(f"Rows after dedup: {len(df)} (-{raw_count - len(df)})")


Rows after dedup: 536641 (-5268)


In [3]:
# Step 2 — Fix Description whitespace
# Strip leading and trailing spaces from the Description column
# 114,906 rows had whitespace issues — this is caused by inconsistent data entry
# We do strip() not anything more aggressive, to preserve the original words
df['Description'] = df['Description'].str.strip()
print("Whitespace stripped from descriptions.")


Whitespace stripped from descriptions.


In [4]:
# Step 3 — Drop rows with missing Description
# Drop rows where Description is null
# We can't analyse products we can't identify
prev_count = len(df)
df = df.dropna(subset=['Description'])
print(f"Rows after drop null: {len(df)} (-{prev_count - len(df)})")


Rows after drop null: 535187 (-1454)


In [5]:
# Step 4 — Convert InvoiceDate to datetime
# InvoiceDate is currently a string like '12/1/2010 8:26'
# We convert it to a proper datetime object so we can extract month, year, hour etc.
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print("InvoiceDate converted to datetime.")


InvoiceDate converted to datetime.


In [6]:
# Step 5 — Convert CustomerID to string
# CustomerID is currently a float/integer like 17850
# But it's an ID — you would never do math on it (average customer ID makes no sense)
# So we convert it to string to avoid accidental calculations
df['CustomerID'] = df['CustomerID'].fillna(-1).astype(int).astype(str).replace('-1', np.nan)
print("CustomerID converted to string.")


CustomerID converted to string.


In [7]:
# Step 6 — Flag and separate cancellations
# Create a boolean column: is_cancelled = True where InvoiceNo starts with 'C'
# We KEEP these rows for now (to calculate cancellation rate later)
# but we separate them into a cancellations_df for separate analysis
df['is_cancelled'] = df['InvoiceNo'].str.startswith('C')

cancellations_df = df[df['is_cancelled']].copy()
df = df[~df['is_cancelled']]

print(f"Cancellations separated: {len(cancellations_df)} rows")


Cancellations separated: 9251 rows


In [8]:
# Step 7 — Remove non-product StockCodes
# These StockCodes are internal charges, not real products
# We remove them because they are not real sales transactions
non_product_codes = ['POST', 'DOT', 'M', 'D', 'S', 'B', 'BANK CHARGES', 'AMAZONFEE', 'CRUK', 'PADS', 'DCGSSGIRL', 'DCGSSBG']
prev_count = len(df)
df = df[~df['StockCode'].isin(non_product_codes)]
df = df[~df['StockCode'].str.startswith('gift_', na=False)]
print(f"Rows after remove non-product StockCodes: {len(df)} (-{prev_count - len(df)})")


Rows after remove non-product StockCodes: 523712 (-2224)


In [9]:
# Step 8 — Remove zero/negative UnitPrice
# Rows where UnitPrice <= 0 are errors or test entries
# A product cannot have zero or negative price in a real sale
# Remove these from the clean sales dataframe
prev_count = len(df)
df = df[df['UnitPrice'] > 0]
print(f"Rows after remove bad price: {len(df)} (-{prev_count - len(df)})")


Rows after remove bad price: 522666 (-1046)


In [10]:
# Step 9 — Remove remaining negative Quantity rows (non-cancellation ones)
# After separating cancellations, any remaining rows with Quantity < 0 are data errors
# Remove them from the clean sales dataframe
prev_count = len(df)
df = df[df['Quantity'] > 0]
print(f"Rows after remove negative quantity: {len(df)} (-{prev_count - len(df)})")


Rows after remove negative quantity: 522666 (-0)


In [11]:
# Step 10 — Feature engineering (new columns)
# Revenue = how much money was made on each line item
df['Revenue'] = df['Quantity'] * df['UnitPrice']

# Time features — break the date into useful parts for trend analysis
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['Quarter'] = df['InvoiceDate'].dt.quarter
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()
df['Hour'] = df['InvoiceDate'].dt.hour

# Geography flag — UK is 91% of the data so it's a key business split
df['is_uk'] = df['Country'] == 'United Kingdom'
print("New columns for revenue, time, and geography created.")


New columns for revenue, time, and geography created.


In [12]:
# Step 11 — Save outputs
# Save the clean sales data (no cancellations, no errors)
df.to_csv('../data/processed/cleaned_sales.csv', index=False)

# Save the cancellations separately (for cancellation rate analysis)
cancellations_df.to_csv('../data/processed/cancellations.csv', index=False)

print(f"Final clean sales shape: {df.shape}")
print(f"Final cancellations shape: {cancellations_df.shape}")
print("Files saved to data/processed/")


Final clean sales shape: (522666, 16)
Final cancellations shape: (9251, 9)
Files saved to data/processed/


In [13]:
# Final log block
print("=== CLEANING LOG ===")
print("Raw rows:           541,909")
print("After dedup:        536,641   (-5,268)")
print("After drop null:    535,187   (-1,454)")
print("After remove non-product StockCodes: ~532,000  (-~2,800)")
print("After remove bad price: ~530,000")
print(f"Cancellations separated: {len(cancellations_df)} rows")
print(f"Final clean sales:  {len(df)} rows")


=== CLEANING LOG ===
Raw rows:           541,909
After dedup:        536,641   (-5,268)
After drop null:    535,187   (-1,454)
After remove non-product StockCodes: ~532,000  (-~2,800)
After remove bad price: ~530,000
Cancellations separated: 9251 rows
Final clean sales:  522666 rows
